In [1]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

c:\Users\xyliz\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


2.11.0+cu128 True NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.6/12.6 MB 1.6 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/12.6 MB 1.6 MB/s eta 0:00:07
   -------- ------------------------------- 2.6/12.6 MB 1.8 MB/s eta 0:00:06
   --------- ------------------------------ 2.9/12.6 MB 1.7 MB/s eta 0:00:06
   ----------- ---------------------------- 3.7/12.6 MB 1.9 MB/s eta 0:00:05
   ------------- -------------------------- 4.2/12.6 MB 2.0 MB/s eta 0:00:05
   -------------- ------------------------- 4.5/12.6 MB 2.0 MB/s eta 0:00:05
   --------------- ------------------------ 5.0/12.6 MB 2.0 MB/s eta 0:00:04
   ---


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random

def load_pairs(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            num, text = line.split("\t")
            pairs.append((num, text))
    return pairs

num_en = load_pairs("num2txt.txt")
num_ko = load_pairs("num2kor.txt")

print(len(num_en), len(num_ko))
for p in random.sample(num_en, 3):
    print(p)
for p in random.sample(num_ko, 3):
    print(p)

def tok_num(s):
    return list(s)

def tok_en(s):
    return s.split()

def tok_ko(s):
    return list(s)

print(tok_num("12345"))
print(tok_en("twelve thousand three hundred forty five"))
print(tok_ko("만 이천삼백사십오"))

en_vocab = sorted({w for _, t in num_en for w in tok_en(t)})
ko_vocab = sorted({c for _, t in num_ko for c in tok_ko(t)})
print(len(en_vocab), en_vocab)
print(len(ko_vocab), ko_vocab)
print(max(len(tok_en(t)) for _, t in num_en), max(len(tok_ko(t)) for _, t in num_ko))

100000 100000
('29043691', 'twenty nine million forty three thousand six hundred ninety one')
('1781342899', 'one billion seven hundred eighty one million three hundred forty two thousand eight hundred ninety nine')
('161176', 'one hundred sixty one thousand one hundred seventy six')
('8359681', '팔백삼십오만 구천육백팔십일')
('1051950507', '십억 오천백구십오만 오백칠')
('1130905', '백십삼만 구백오')
['1', '2', '3', '4', '5']
['twelve', 'thousand', 'three', 'hundred', 'forty', 'five']
['만', ' ', '이', '천', '삼', '백', '사', '십', '오']
32 ['billion', 'eight', 'eighteen', 'eighty', 'eleven', 'fifteen', 'fifty', 'five', 'forty', 'four', 'fourteen', 'hundred', 'million', 'nine', 'nineteen', 'ninety', 'one', 'seven', 'seventeen', 'seventy', 'six', 'sixteen', 'sixty', 'ten', 'thirteen', 'thirty', 'thousand', 'three', 'twelve', 'twenty', 'two', 'zero']
16 [' ', '구', '만', '백', '사', '삼', '십', '억', '영', '오', '육', '이', '일', '천', '칠', '팔']
16 21


In [4]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3

class Vocab:
    def __init__(self, tokens, joiner):
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + sorted(set(tokens))
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.joiner = joiner

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [SOS] + [self.stoi.get(t, UNK) for t in tokens] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return self.joiner.join(out)

num_vocab = Vocab("0123456789", "")
en_vocab = Vocab([w for _, t in num_en for w in tok_en(t)], " ")
ko_vocab = Vocab([c for _, t in num_ko for c in tok_ko(t)], "")

print(len(num_vocab), len(en_vocab), len(ko_vocab))

ids = en_vocab.encode(tok_en("twelve thousand three hundred forty five"))
print(ids)
print(en_vocab.decode(ids))

ids = ko_vocab.encode(tok_ko("만 이천삼백사십오"))
print(ids)
print(ko_vocab.decode(ids))

ids = num_vocab.encode(tok_num("12345"))
print(ids)
print(num_vocab.decode(ids))

14 36 20
[1, 32, 30, 31, 15, 12, 11, 2]
twelve thousand three hundred forty five
[1, 6, 4, 15, 17, 9, 7, 8, 10, 13, 2]
만 이천삼백사십오
[1, 5, 6, 7, 8, 9, 2]
12345


In [5]:
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

random.seed(42)

LANGS = {
    "num": (tok_num, num_vocab),
    "en": (tok_en, en_vocab),
    "ko": (tok_ko, ko_vocab),
}

num2en = dict(num_en)
num2ko = dict(num_ko)
shared = sorted(set(num2en) & set(num2ko))
en_ko = [(num2en[n], num2ko[n]) for n in shared]
print(len(en_ko))

def split(pairs, ratio=(0.8, 0.1, 0.1)):
    pairs = pairs[:]
    random.shuffle(pairs)
    a = int(len(pairs) * ratio[0])
    b = a + int(len(pairs) * ratio[1])
    return pairs[:a], pairs[a:b], pairs[b:]

base = {
    ("num", "en"): split(num_en),
    ("num", "ko"): split(num_ko),
    ("en", "ko"): split(en_ko),
}

data = {}
for (a, b), (tr, va, te) in base.items():
    data[(a, b)] = (tr, va, te)
    data[(b, a)] = tuple([(y, x) for x, y in part] for part in (tr, va, te))

for k, (tr, va, te) in data.items():
    print(k, len(tr), len(va), len(te), tr[0])

class PairDataset(Dataset):
    def __init__(self, pairs, src, tgt):
        s_tok, s_voc = LANGS[src]
        t_tok, t_voc = LANGS[tgt]
        self.data = [
            (torch.tensor(s_voc.encode(s_tok(x))), torch.tensor(t_voc.encode(t_tok(y))))
            for x, y in pairs
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

def collate(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, tgt

ds = PairDataset(data[("num", "en")][0], "num", "en")
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate)
src, tgt = next(iter(loader))
print(src.shape, tgt.shape)
print(src)
print(tgt)
for s, t in zip(src, tgt):
    print(num_vocab.decode(s.tolist()), "->", en_vocab.decode(t.tolist()))

12738
('num', 'en') 80000 10000 10000 ('962', 'nine hundred sixty two')
('en', 'num') 80000 10000 10000 ('nine hundred sixty two', '962')
('num', 'ko') 80000 10000 10000 ('12', '십이')
('ko', 'num') 80000 10000 10000 ('십이', '12')
('en', 'ko') 10190 1273 1275 ('eight thousand one hundred forty two', '팔천백사십이')
('ko', 'en') 10190 1273 1275 ('팔천백사십이', 'eight thousand one hundred forty two')
torch.Size([4, 12]) torch.Size([4, 17])
tensor([[ 1,  7, 11,  5,  7, 10,  6,  8,  6,  9, 13,  2],
        [ 1,  9,  7,  5, 10,  8, 13,  9,  4,  9,  2,  0],
        [ 1, 12,  7,  9,  8,  8,  4, 13,  5,  2,  0,  0],
        [ 1,  9,  4, 10,  7,  7,  4,  4,  6,  2,  0,  0]])
tensor([[ 1, 31,  4, 21, 15, 28, 16, 24, 15, 33, 13, 30, 34, 15, 10, 17,  2],
        [ 1, 11, 15, 29, 20, 16, 24, 15, 12, 17, 30, 11, 15, 11,  2,  0,  0],
        [ 1,  7, 31, 16, 11, 15, 12, 13, 30, 19, 20,  2,  0,  0,  0,  0,  0],
        [ 1, 10, 16, 24, 15, 29, 31, 30, 34,  2,  0,  0,  0,  0,  0,  0,  0]])
3713624259 -> three billio